# nb3 — Phase 1: Baseline LoRA BARTpho & Đánh giá 3 chỉ số

Notebook thứ tư và là notebook chốt hạ của Phase 1 (`DESIGN.md §7`):
- Huấn luyện mô hình cơ sở seq2seq **BARTpho-syllable** (`vinai/bartpho-syllable`, ~400M params) kết hợp **LoRA fp16** trên Kaggle T4.
- Triển khai **2 Run đối chứng** theo Quyết định 22/09 (`DESIGN.md §1` & `implementation_plan.md`):
  - **Run 1 (Pure Baseline)**: Huấn luyện thuần trên `VSEC-train` gốc (đo mức độ over-correction nguyên bản).
  - **Run 2 (Baseline + Augmentation)**: Huấn luyện với tập bổ sung câu sạch (từ train `corrected_text`) + câu nhiễu (sinh bởi `noise_model.json` của `nb2`) để kiểm chứng khả năng giảm over-correction.
- **Đánh giá tách bạch 3 bộ chỉ số bắt buộc** (`PROJECT.md §6`):
  1. **Detection**: Precision, Recall, F1 của việc phát hiện vị trí âm tiết lỗi.
  2. **Correction Accuracy**: Tỷ lệ sửa đúng tại các vị trí đã detect đúng.
  3. **Over-correction Rate**: Tỷ lệ âm tiết vốn dĩ viết đúng nhưng bị mô hình tự ý thay đổi (và tỷ lệ giữ nguyên câu sạch).
  - Đánh giá trên cả tập **VSEC-val** (927 câu) và **Test 6.000 câu** (`test_aligned.jsonl` từ `nb1`).
  - Phân tích phân tầng (Stratified analysis): Non-word vs Real-word (dùng `syllable_table.json` từ `nb2`).

**Input**:
- Từ `nb0`: `vsec_train.jsonl`, `vsec_val.jsonl`
- Từ `nb1`: `test_aligned.jsonl`
- Từ `nb2`: `noise_model.json`, `syllable_table.json`

**Quy tắc chống rò rỉ dữ liệu (Anti-leakage checklist - DESIGN.md §9)**:
- Nguồn câu sạch và câu nhiễu sinh augmentation **chỉ lấy từ `vsec_train.jsonl`** (`split == 'train'`).
- Tuyệt đối không nạp thông tin hay nhãn của `val` hoặc `test` vào quá trình huấn luyện.
- Cố định seed 42, kiểm thử sanity metrics bằng test case nhân tạo trước khi tính trên tập thật.


## 0. Cấu hình & Tự dò 3 nguồn Input (DESIGN.md §7)

Gom toàn bộ siêu tham số tại một vị trí để dễ điều chỉnh và kiểm soát:
- `SEED = 42`: Tái lập kết quả.
- `MODEL_NAME = 'vinai/bartpho-syllable'`: Mô hình Pre-trained Seq2Seq tiếng Việt mức âm tiết.
- `LORA_R = 16`, `LORA_ALPHA = 32`, `LORA_DROPOUT = 0.05`: Cấu hình LoRA PEFT.
- `BATCH_SIZE = 8`, `GRAD_ACCUM = 4`, `LR = 2e-4`, `EPOCHS = 6`: Effective batch size = 32.
- `FP16 = True`: Tăng tốc và tiết kiệm VRAM trên Kaggle T4.
- `AUG_CLEAN_RATIO = 0.3`, `AUG_NOISE_RATIO = 0.3`: Tỷ lệ pha 30% câu sạch + 30% câu nhiễu cho Run 2.


In [1]:
import os
import sys
import json
import random
import datetime
import unicodedata
from pathlib import Path
import collections

# Cài đặt thư viện cần thiết nếu chạy trên Kaggle
try:
    import peft
    import sentencepiece
except ImportError:
    print('Cài đặt peft và sentencepiece...')
    os.system('pip install -q peft sentencepiece')
    os.system('pip uninstall -y torchao')

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

SEED = 42
set_seed(SEED)

MODEL_NAME = 'vinai/bartpho-syllable'
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]

BATCH_SIZE = 8
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
EPOCHS = 6
FP16 = torch.cuda.is_available()
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

AUG_CLEAN_RATIO = 0.3
AUG_NOISE_RATIO = 0.3
EVAL_BEAM_SIZE = 1  # Greedy để eval nhanh trên 6.000 câu test

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
        
    found = {}
    for p in candidates:
        name = p.name
        if name in ['vsec_train.jsonl', 'vsec_val.jsonl', 'test_aligned.jsonl', 'noise_model.json', 'syllable_table.json']:
            if name not in found:
                found[name] = p
                
    missing = [req for req in ['vsec_train.jsonl', 'vsec_val.jsonl', 'noise_model.json', 'syllable_table.json'] if req not in found]
    if missing:
        raise FileNotFoundError(f'Thiếu các tệp đầu vào bắt buộc: {missing}. Hãy kiểm tra Input Datasets trên Kaggle hoặc thư mục ./out!')
    return found

INPUT_FILES = find_required_inputs()
print('=== TỰ DÒ INPUT HOÀN TẤT ===')
for k, v in INPUT_FILES.items():
    print(f'  {k:20s}: {v}')
print(f'Device: {"CUDA " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"} | FP16={FP16}')


=== TỰ DÒ INPUT HOÀN TẤT ===
  vsec_train.jsonl    : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/vsec_train.jsonl
  vsec_val.jsonl      : /kaggle/input/notebooks/cquangnguynl/nb0-data-pre/vsec_val.jsonl
  test_aligned.jsonl  : /kaggle/input/notebooks/cquangnguynl/nb1-align-annotate/test_aligned.jsonl
  noise_model.json    : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/noise_model.json
  syllable_table.json : /kaggle/input/notebooks/cquangnguynl/nb2-pilot-dict-noise/syllable_table.json
Device: CUDA Tesla T4 | FP16=True


## 1. Cell hàm dùng chung (`SHARED_CELLS_VERSION = 'align-v1'`) & Metric Calculator

Được sao chép nguyên vẹn từ `nb1` và `nb2` để đảm bảo chuẩn hóa Unicode NFC và phân tách token canonical hoàn toàn thống nhất.
Kèm theo module tính toán **3 bộ chỉ số đánh giá** (`evaluate_predictions`) độc lập và chuẩn xác.


In [2]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)


SHARED_CELLS_VERSION: align-v1


### Hàm đánh giá chuẩn 3 lớp chỉ số (Detection, Correction, Over-correction)

Quy ước ánh xạ vị trí sửa đổi (`src_span`):
- Khi mô hình dự đoán `pred_text`, ta align `src ↔ pred` để trích xuất `pred_edits`.
- Đối chiếu tập vị trí sửa của mô hình với vị trí lỗi thực tế (`gold_edits` trích xuất từ `src ↔ gold`):
  - **Detection True Positive (TP)**: Vị trí token nguồn mà cả gold và model đều đánh dấu cần sửa.
  - **Detection False Positive (FP - Over-correction)**: Token nguồn vốn dĩ viết đúng nhưng mô hình tự ý sửa đổi.
  - **Detection False Negative (FN)**: Token lỗi nhưng mô hình bỏ qua không sửa.
  - **Correction Accuracy**: Trong số các token detect đúng (TP), tỷ lệ sửa ra token hoàn toàn trùng khớp với gold.
  - **Over-correction Rate**: $FP / \text{Tổng số token đúng trong câu nguồn}$.
  - **Clean Sentence Accuracy**: Tỷ lệ giữ nguyên 100% đối với các câu ban đầu không có lỗi.


In [3]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')


Sanity check hàm evaluate_predictions PASS 100%!


## 2. Chuẩn bị dữ liệu huấn luyện: VSEC thuần (Run 1) vs Augmentation (Run 2)

Tuân thủ nghiêm ngặt **Quy tắc chống rò rỉ dữ liệu (PROJECT.md §5 & DESIGN.md §9)**:
- Nạp `vsec_train.jsonl` (8.343 câu) và `vsec_val.jsonl` (927 câu).
- Tải `noise_model.json` và `syllable_table.json` từ `nb2`.
- **Tập Train Run 1 (Pure Baseline)**: 100% câu từ `vsec_train.jsonl`.
- **Tập Train Run 2 (Augmentation)**:
  - Sinh câu sạch (Identity mapping: `text = corrected_text` từ tập train) với tỷ lệ `AUG_CLEAN_RATIO = 0.3` (~2.500 câu).
  - Sinh câu nhiễu (Dùng bộ sinh `generate_noisy` từ `noise_model.json` áp lên `corrected_text` tập train) với tỷ lệ `AUG_NOISE_RATIO = 0.3` (~2.500 câu).
  - **Assert chắc chắn**: Mọi câu được lấy đều có `split == 'train'`.


In [4]:
train_records = load_jsonl(INPUT_FILES['vsec_train.jsonl'])
val_records = load_jsonl(INPUT_FILES['vsec_val.jsonl'])
test_records = load_jsonl(INPUT_FILES['test_aligned.jsonl']) if 'test_aligned.jsonl' in INPUT_FILES else []

print(f'Số lượng câu nạp vào — Train: {len(train_records)} | Val: {len(val_records)} | Test: {len(test_records)}')

# Nạp noise model và bảng âm tiết từ nb2
noise_model_dict = json.loads(Path(INPUT_FILES['noise_model.json']).read_text(encoding='utf-8'))
syllable_table_dict = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(syllable_table_dict['entries'])

# Tái lập hàm generate_noisy từ noise_model.json của nb2
P_EMPIRICAL = noise_model_dict['config']['p_empirical']
MAX_ERRORS_PER_SENT = noise_model_dict['config']['max_errors_per_sent']
QWERTY_ADJ = noise_model_dict['config']['qwerty_adj']
VOWEL_MAP = noise_model_dict['config']['vowel_map']
REGIONAL_SWAPS = [tuple(x) for x in noise_model_dict['config']['regional_swaps']]
TONE_MARKS = [chr(int(x, 16)) for x in noise_model_dict['config']['tone_marks']]
CONFUSION = noise_model_dict['confusion']
ERROR_COUNT_DIST = {int(k): v for k, v in noise_model_dict['error_count_dist'].items()}
ERROR_VOCAB = {e for errs in CONFUSION.values() for e in errs}

def char_lev_at_most_1(a, b):
    if abs(len(a) - len(b)) > 1: return False
    if len(a) < len(b): a, b = b, a
    i = j = diff = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            i += 1; j += 1
        else:
            diff += 1
            if diff > 1: return False
            if len(a) == len(b): i += 1; j += 1
            else: i += 1
    if i < len(a) or j < len(b): diff += 1
    return diff <= 1

def rule_candidates(syl):
    cands = {}
    # regional
    for src, dst in REGIONAL_SWAPS:
        if syl.startswith(src):
            rest = syl[len(src):]
            if rest and rest[0] in set('aăâeêioôơuưy'):
                cands[dst + rest] = 'regional'
    # tone
    d = unicodedata.normalize('NFD', syl)
    for mark in TONE_MARKS:
        if mark in d:
            for m in TONE_MARKS:
                if m != mark:
                    cands[unicodedata.normalize('NFC', d.replace(mark, m))] = 'tone'
    # vowel
    for i, ch in enumerate(syl):
        for alt in VOWEL_MAP.get(ch, ''):
            if alt != ch:
                cands[syl[:i] + alt + syl[i+1:]] = 'vowel'
    # keyboard
    for i, ch in enumerate(syl):
        if ch.isascii() and ch.isalpha():
            for rep in QWERTY_ADJ.get(ch.lower(), ''):
                cands[syl[:i] + rep + syl[i+1:]] = 'keyboard'
            cands[syl[:i] + syl[i+1:]] = 'keyboard'
            cands[syl[:i] + ch + syl[i:]] = 'keyboard'
    return cands

def accepted_rule_candidates(low_tok, syll_set):
    cands = {}
    for c, src in rule_candidates(low_tok).items():
        if c != low_tok and (c in syll_set or c in ERROR_VOCAB or char_lev_at_most_1(c, low_tok)):
            cands[c] = 'rule:' + src
    return cands

def generate_noisy(clean_text, rng, syll_set):
    toks = canon_tokenize(clean_text)
    cand_cache = {}
    eligible = []
    for i, tok in enumerate(toks):
        if not is_word_token(tok): continue
        low = nfc_normalize(tok).lower()
        opts = {}
        for e in CONFUSION.get(low, {}):
            if e != low: opts[e] = 'empirical'
        for c, src in accepted_rule_candidates(low, syll_set).items():
            opts.setdefault(c, src)
        if opts:
            eligible.append(i)
            cand_cache[i] = (low, opts)
    if not eligible: return None
    k = min(rng.choices(list(ERROR_COUNT_DIST), weights=list(ERROR_COUNT_DIST.values()), k=1)[0], len(eligible))
    out = list(toks)
    for i in sorted(rng.sample(eligible, k)):
        low, opts = cand_cache[i]
        emp = {c: CONFUSION[low][c] for c, s in opts.items() if s == 'empirical'}
        if emp and rng.random() < P_EMPIRICAL:
            pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
        else:
            rules = [c for c, s in opts.items() if s.startswith('rule:')]
            if rules: pick = rng.choice(rules)
            else: pick = rng.choices(list(emp), weights=list(emp.values()), k=1)[0]
        cand = pick[:1].upper() + pick[1:] if toks[i][:1].isupper() else pick
        out[i] = cand
    return ' '.join(out)

# TẠO TẬP HUẤN LUYỆN CHO RUN 1 VÀ RUN 2
rng = random.Random(SEED)
for r in train_records:
    assert r.get('split') == 'train', f'Data leakage alert: record {r.get("row_id")} không phải split=train'

# Run 1: Pure VSEC train
train_data_run1 = [{'text': r['text'], 'corrected_text': r['corrected_text']} for r in train_records]

# Run 2: Train + Augmentation
n_clean = int(len(train_records) * AUG_CLEAN_RATIO)
n_noise = int(len(train_records) * AUG_NOISE_RATIO)

clean_samples = rng.sample(train_records, n_clean)
clean_aug = [{'text': r['corrected_text'], 'corrected_text': r['corrected_text']} for r in clean_samples]

noise_samples = rng.sample(train_records, min(n_noise * 2, len(train_records)))
noisy_aug = []
for r in noise_samples:
    noisy_str = generate_noisy(r['corrected_text'], rng, SYLL_SET)
    if noisy_str is not None and noisy_str != r['corrected_text']:
        noisy_aug.append({'text': noisy_str, 'corrected_text': r['corrected_text']})
    if len(noisy_aug) >= n_noise:
        break

train_data_run2 = train_data_run1 + clean_aug + noisy_aug

print(f'=== TẬP DỮ LIỆU HUẤN LUYỆN ===')
print(f'Run 1 (Pure Baseline) : {len(train_data_run1)} câu')
print(f'Run 2 (With Augment)  : {len(train_data_run2)} câu (+{len(clean_aug)} câu sạch, +{len(noisy_aug)} câu nhiễu)')


Số lượng câu nạp vào — Train: 8343 | Val: 927 | Test: 5983
=== TẬP DỮ LIỆU HUẤN LUYỆN ===
Run 1 (Pure Baseline) : 8343 câu
Run 2 (With Augment)  : 13347 câu (+2502 câu sạch, +2502 câu nhiễu)


## 3. Tokenization với BARTpho-syllable & PyTorch Dataset

Sử dụng tokenizer `vinai/bartpho-syllable` với mã hóa BPE âm tiết:
- Đầu vào: `text` (câu có lỗi hoặc câu sạch).
- Nhãn: `corrected_text` (câu đã sửa chuẩn).
- Padding token trong `labels` được gán giá trị `-100` để CrossEntropyLoss tự động bỏ qua.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SpellingDataset(torch.utils.data.Dataset):
    def __init__(self, data_list, tokenizer, max_src_len=MAX_SOURCE_LEN, max_tgt_len=MAX_TARGET_LEN):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src_enc = self.tokenizer(
            item['text'],
            max_length=self.max_src_len,
            truncation=True,
            padding=False
        )
        tgt_enc = self.tokenizer(
            item['corrected_text'],
            max_length=self.max_tgt_len,
            truncation=True,
            padding=False
        )
        return {
            'input_ids': src_enc['input_ids'],
            'attention_mask': src_enc['attention_mask'],
            'labels': tgt_enc['input_ids']
        }

val_dataset = SpellingDataset(val_records, tokenizer)
ds_run1 = SpellingDataset(train_data_run1, tokenizer)
ds_run2 = SpellingDataset(train_data_run2, tokenizer)

print(f'Dataset sẵn sàng: Train R1={len(ds_run1)}, Train R2={len(ds_run2)}, Val={len(val_dataset)}')


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

Dataset sẵn sàng: Train R1=8343, Train R2=13347, Val=927


## 4. Khởi tạo mô hình BARTpho + LoRA fp16

Tối ưu hóa tài nguyên cho GPU T4 (15GB):
- Nạp trọng số cơ sở ở định dạng `fp16`.
- Bật `gradient_checkpointing_enable()` để tiết kiệm bộ nhớ kích hoạt.
- Cấu hình LoRA PEFT: gán adapter vào các ma trận chiếu chú ý `q_proj`, `v_proj`, `k_proj`, `out_proj`.
- Chỉ cập nhật ~0.5% - 1% tổng số tham số, giảm thiểu nguy cơ catastrophic forgetting và tăng tốc hội tụ.


In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
def build_lora_bartpho():
    print(f'Nạp mô hình cơ sở {MODEL_NAME} (torch_dtype=float16)...')
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if FP16 else torch.float32
    )
    model.gradient_checkpointing_enable()
    
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none"
    )
    lora_model = get_peft_model(model, peft_config)
    trainable_params, all_param = lora_model.get_nb_trainable_parameters()
    print(f'Trainable params: {trainable_params:,} / {all_param:,} ({100 * trainable_params / all_param:.2f}%)')
    return lora_model

# Kiểm tra khởi tạo thử nghiệm
_m = build_lora_bartpho()
del _m
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Nạp mô hình cơ sở vinai/bartpho-syllable (torch_dtype=float16)...


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Trainable params: 4,718,592 / 482,514,944 (0.98%)


## 5. Training Run 1 — Pure Baseline (Không Augmentation)

Huấn luyện mô hình cơ sở thuần túy trên `vsec_train.jsonl` để đo mức độ over-correction và các chỉ số gốc.


In [8]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

def get_training_args(output_subdir):
    return Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR / output_subdir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.05,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        fp16=FP16,
        predict_with_generate=False,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="loss"
    )

print('=== BẮT ĐẦU HUẤN LUYỆN RUN 1 (PURE BASELINE) ===')
model_run1 = build_lora_bartpho()
args_run1 = get_training_args('checkpoints_run1_pure')

trainer_run1 = Seq2SeqTrainer(
    model=model_run1,
    args=args_run1,
    train_dataset=ds_run1,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=collator
)

trainer_run1.train()

# Lưu adapter Run 1
adapter_run1_dir = OUTPUT_DIR / 'lora_adapter_run1'
model_run1.save_pretrained(str(adapter_run1_dir))
print(f'Đã lưu adapter Run 1 vào {adapter_run1_dir}')


=== BẮT ĐẦU HUẤN LUYỆN RUN 1 (PURE BASELINE) ===
Nạp mô hình cơ sở vinai/bartpho-syllable (torch_dtype=float16)...


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Trainable params: 4,718,592 / 482,514,944 (0.98%)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,2.019883,0.344238
2,1.234104,0.225586
3,1.086843,0.179443
4,0.951631,0.169556
5,0.885291,0.162964
6,0.872642,0.164185


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Đã lưu adapter Run 1 vào /kaggle/working/lora_adapter_run1


## 6. Training Run 2 — Baseline + Augmentation

Huấn luyện mô hình thứ hai trên tập dữ liệu đã pha thêm 30% câu sạch + 30% câu nhiễu với cùng siêu tham số.


In [9]:
print('=== BẮT ĐẦU HUẤN LUYỆN RUN 2 (BASELINE + AUGMENTATION) ===')
model_run2 = build_lora_bartpho()
args_run2 = get_training_args('checkpoints_run2_augmented')

trainer_run2 = Seq2SeqTrainer(
    model=model_run2,
    args=args_run2,
    train_dataset=ds_run2,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=collator
)

trainer_run2.train()

# Lưu adapter Run 2
adapter_run2_dir = OUTPUT_DIR / 'lora_adapter_run2'
model_run2.save_pretrained(str(adapter_run2_dir))
print(f'Đã lưu adapter Run 2 vào {adapter_run2_dir}')


=== BẮT ĐẦU HUẤN LUYỆN RUN 2 (BASELINE + AUGMENTATION) ===
Nạp mô hình cơ sở vinai/bartpho-syllable (torch_dtype=float16)...


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable params: 4,718,592 / 482,514,944 (0.98%)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.355557,0.292725
2,1.065527,0.234863
3,0.882941,0.199463
4,0.804382,0.183105
5,0.750748,0.175659
6,0.716786,0.181152


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Đã lưu adapter Run 2 vào /kaggle/working/lora_adapter_run2


## 7. Inference & Đánh giá 3 bộ chỉ số trên VSEC-val và Test 6k

Chạy bộ giải mã sinh câu (Generation) trên:
- **Tập VSEC-val (927 câu)**: Đánh giá trên dữ liệu gold có chú giải chi tiết.
- **Tập Test 6.000 câu** (`test_aligned.jsonl` từ `nb1`): Đánh giá trên tập dữ liệu độc lập ngoài phân bố VSEC.


In [10]:
def batch_generate(model, tokenizer, texts, batch_size=16, beam_size=EVAL_BEAM_SIZE):
    model.eval()
    device = next(model.parameters()).device
    preds = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                max_length=MAX_TARGET_LEN,
                num_beams=beam_size,
                early_stopping=True if beam_size > 1 else False
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([nfc_normalize(d) for d in decoded])
        
        if (i // batch_size) % 50 == 0:
            print(f'  Generated {min(i + batch_size, len(texts))}/{len(texts)} sentences...')
    return preds

val_texts = [r['text'] for r in val_records]
test_texts = [r['text'] for r in test_records] if test_records else []

print('--- INFERENCE RUN 1 (PURE BASELINE) ---')
preds_val_r1 = batch_generate(model_run1, tokenizer, val_texts)
preds_test_r1 = batch_generate(model_run1, tokenizer, test_texts) if test_texts else []

print('--- INFERENCE RUN 2 (AUGMENTED) ---')
preds_val_r2 = batch_generate(model_run2, tokenizer, val_texts)
preds_test_r2 = batch_generate(model_run2, tokenizer, test_texts) if test_texts else []

print('Inference hoàn tất trên cả Val và Test!')


--- INFERENCE RUN 1 (PURE BASELINE) ---
  Generated 16/927 sentences...
  Generated 816/927 sentences...
  Generated 16/5983 sentences...
  Generated 816/5983 sentences...
  Generated 1616/5983 sentences...
  Generated 2416/5983 sentences...
  Generated 3216/5983 sentences...
  Generated 4016/5983 sentences...
  Generated 4816/5983 sentences...
  Generated 5616/5983 sentences...
--- INFERENCE RUN 2 (AUGMENTED) ---
  Generated 16/927 sentences...
  Generated 816/927 sentences...
  Generated 16/5983 sentences...
  Generated 816/5983 sentences...
  Generated 1616/5983 sentences...
  Generated 2416/5983 sentences...
  Generated 3216/5983 sentences...
  Generated 4016/5983 sentences...
  Generated 4816/5983 sentences...
  Generated 5616/5983 sentences...
Inference hoàn tất trên cả Val và Test!


## 8. Báo cáo đối chiếu Run 1 vs Run 2 & Phân tích Stratified

Tính toán toàn diện và in bảng so sánh trực diện:
- **Detection**: Precision, Recall, F1.
- **Correction**: Accuracy tại các vị trí TP.
- **Over-correction Rate**: % token sạch bị sửa sai (và tỷ lệ giữ nguyên câu sạch).


In [11]:
eval_val_r1 = evaluate_predictions(val_records, preds_val_r1, SYLL_SET)
eval_val_r2 = evaluate_predictions(val_records, preds_val_r2, SYLL_SET)

eval_test_r1 = evaluate_predictions(test_records, preds_test_r1, SYLL_SET) if test_records else None
eval_test_r2 = evaluate_predictions(test_records, preds_test_r2, SYLL_SET) if test_records else None

def print_metrics_table(val_r1, val_r2, test_r1=None, test_r2=None):
    print('='*86)
    print(f'{"CHỈ SỐ ĐÁNH GIÁ":<30s} | {"VAL: Run 1":<12s} | {"VAL: Run 2":<12s} | {"TEST: Run 1":<12s} | {"TEST: Run 2":<12s}')
    print('-'*86)
    
    rows = [
        ("Detection Precision", f"{val_r1['detection']['precision']:.2%}", f"{val_r2['detection']['precision']:.2%}",
         f"{test_r1['detection']['precision']:.2%}" if test_r1 else "N/A", f"{test_r2['detection']['precision']:.2%}" if test_r2 else "N/A"),
        ("Detection Recall", f"{val_r1['detection']['recall']:.2%}", f"{val_r2['detection']['recall']:.2%}",
         f"{test_r1['detection']['recall']:.2%}" if test_r1 else "N/A", f"{test_r2['detection']['recall']:.2%}" if test_r2 else "N/A"),
        ("Detection F1-Score", f"{val_r1['detection']['f1']:.2%}", f"{val_r2['detection']['f1']:.2%}",
         f"{test_r1['detection']['f1']:.2%}" if test_r1 else "N/A", f"{test_r2['detection']['f1']:.2%}" if test_r2 else "N/A"),
        ("Correction Accuracy (at TP)", f"{val_r1['correction']['accuracy']:.2%}", f"{val_r2['correction']['accuracy']:.2%}",
         f"{test_r1['correction']['accuracy']:.2%}" if test_r1 else "N/A", f"{test_r2['correction']['accuracy']:.2%}" if test_r2 else "N/A"),
        ("Over-correction Rate", f"{val_r1['over_correction']['rate']:.2%}", f"{val_r2['over_correction']['rate']:.2%}",
         f"{test_r1['over_correction']['rate']:.2%}" if test_r1 else "N/A", f"{test_r2['over_correction']['rate']:.2%}" if test_r2 else "N/A"),
        ("Clean Sent Retention Rate", f"{val_r1['over_correction']['clean_retention_rate']:.2%}", f"{val_r2['over_correction']['clean_retention_rate']:.2%}",
         f"{test_r1['over_correction']['clean_retention_rate']:.2%}" if test_r1 else "N/A", f"{test_r2['over_correction']['clean_retention_rate']:.2%}" if test_r2 else "N/A"),
    ]
    for name, v1, v2, t1, t2 in rows:
        print(f'{name:<30s} | {v1:<12s} | {v2:<12s} | {t1:<12s} | {t2:<12s}')
    print('='*86)

print_metrics_table(eval_val_r1, eval_val_r2, eval_test_r1, eval_test_r2)

print('\n=== PHÂN TÍCH STRATIFIED (VSEC-VAL) ===')
for cat in ['nonword', 'realword']:
    r1_det = eval_val_r1['stratified'][cat]['detected'] / eval_val_r1['stratified'][cat]['gold'] if eval_val_r1['stratified'][cat]['gold'] else 0.0
    r2_det = eval_val_r2['stratified'][cat]['detected'] / eval_val_r2['stratified'][cat]['gold'] if eval_val_r2['stratified'][cat]['gold'] else 0.0
    print(f'Loại lỗi [{cat.upper():8s}] — Gold: {eval_val_r1["stratified"][cat]["gold"]} lỗi')
    print(f'  Run 1 Detection Recall: {r1_det:.2%} | Run 2 Detection Recall: {r2_det:.2%}')

print('\n=== 5 MẪU OVER-CORRECTION (FP) TIÊU BIỂU CỦA RUN 1 ===')
for s in eval_val_r1['sample_overcorrections'][:5]:
    print(f"  Từ đúng: '{s['orig_token']}' -> Bị sửa thành: '{s['pred_token']}'")
    print(f"  Ngữ cảnh: ...{s['context']}...")
    print()


CHỈ SỐ ĐÁNH GIÁ                | VAL: Run 1   | VAL: Run 2   | TEST: Run 1  | TEST: Run 2 
--------------------------------------------------------------------------------------
Detection Precision            | 80.51%       | 79.00%       | 86.03%       | 86.67%      
Detection Recall               | 72.27%       | 70.78%       | 68.77%       | 75.07%      
Detection F1-Score             | 76.16%       | 74.67%       | 76.43%       | 80.45%      
Correction Accuracy (at TP)    | 84.75%       | 86.53%       | 66.20%       | 67.94%      
Over-correction Rate           | 0.75%        | 0.81%        | 1.82%        | 1.88%       
Clean Sent Retention Rate      | 33.33%       | 33.33%       | 86.68%       | 85.91%      

=== PHÂN TÍCH STRATIFIED (VSEC-VAL) ===
Loại lỗi [NONWORD ] — Gold: 296 lỗi
  Run 1 Detection Recall: 75.68% | Run 2 Detection Recall: 74.32%
Loại lỗi [REALWORD] — Gold: 847 lỗi
  Run 1 Detection Recall: 71.07% | Run 2 Detection Recall: 69.54%

=== 5 MẪU OVER-CORRECTION (FP)

## 9. Xuất file đầu ra & Báo cáo tổng hợp (`eval_report.json`)

Lưu các file kết quả để đối chiếu:
- `eval_report.json`: Toàn bộ chỉ số định lượng.
- `predictions_val_run1.jsonl`, `predictions_val_run2.jsonl`.
- `predictions_test_run1.jsonl`, `predictions_test_run2.jsonl`.


In [12]:
def save_predictions_jsonl(records, preds, out_path):
    with open(out_path, 'w', encoding='utf-8') as f:
        for r, p in zip(records, preds):
            item = {
                'row_id': r.get('row_id'),
                'text': r['text'],
                'corrected_text': r['corrected_text'],
                'prediction': p
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_predictions_jsonl(val_records, preds_val_r1, OUTPUT_DIR / 'predictions_val_run1.jsonl')
save_predictions_jsonl(val_records, preds_val_r2, OUTPUT_DIR / 'predictions_val_run2.jsonl')

if test_records:
    save_predictions_jsonl(test_records, preds_test_r1, OUTPUT_DIR / 'predictions_test_run1.jsonl')
    save_predictions_jsonl(test_records, preds_test_r2, OUTPUT_DIR / 'predictions_test_run2.jsonl')

eval_report = {
    'created': RUN_STAMP,
    'notebook': 'nb3_baseline_train_eval',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'config': {
        'seed': SEED,
        'model_name': MODEL_NAME,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'learning_rate': LEARNING_RATE,
        'aug_clean_ratio': AUG_CLEAN_RATIO,
        'aug_noise_ratio': AUG_NOISE_RATIO
    },
    'results': {
        'val_run1_pure': eval_val_r1,
        'val_run2_aug': eval_val_r2,
        'test_run1_pure': eval_test_r1,
        'test_run2_aug': eval_test_r2
    }
}

with open(OUTPUT_DIR / 'eval_report.json', 'w', encoding='utf-8') as f:
    json.dump(eval_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT XUẤT ĐẦU RA ===')
print('Các file đã ghi vào', OUTPUT_DIR)
for fn in ['predictions_val_run1.jsonl', 'predictions_val_run2.jsonl', 'eval_report.json']:
    print(' -', fn)


=== HOÀN TẤT XUẤT ĐẦU RA ===
Các file đã ghi vào /kaggle/working
 - predictions_val_run1.jsonl
 - predictions_val_run2.jsonl
 - eval_report.json
